In [ ]:
# I linked most of my code and file to my drive to have easy access and be able
# transfer everything to the GIT when submitting

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import re

DATA_RAW = Path("/content/drive/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"
FIGURES = DATA_RAW / "figures"
TABLES = DATA_RAW / "tables"

for folder in [DATA_PROCESSED, FIGURES, TABLES]:
    folder.mkdir(parents=True, exist_ok=True)

print("DATA_RAW:", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("FIGURES:", FIGURES)
print("TABLES:", TABLES)

Mounted at /content/drive
DATA_RAW: /content/drive/MyDrive/STAT596/Project596_datafiles
DATA_PROCESSED: /content/drive/MyDrive/STAT596/Project596_datafiles/processed_data
FIGURES: /content/drive/MyDrive/STAT596/Project596_datafiles/figures
TABLES: /content/drive/MyDrive/STAT596/Project596_datafiles/tables


In [ ]:
candidate_files = [
    DATA_RAW / "sd_beach_advisories_clean_2023_latest.csv",
    DATA_RAW / "beach-advisories.csv"
]

beach_path = next((path for path in candidate_files if path.exists()), None)

if beach_path is None:
    raise FileNotFoundError(
        "Could not find sd_beach_advisories_clean_2023_latest.csv or beach-advisories.csv in DATA_RAW."
    )

beach_raw = pd.read_csv(beach_path, low_memory=False)

print("Loaded file:", beach_path.name)
print("Rows:", beach_raw.shape[0])
print("Columns:", beach_raw.shape[1])

display(beach_raw.head())

column_check = pd.DataFrame({
    "column": beach_raw.columns,
    "non_missing": beach_raw.notna().sum().values,
    "missing": beach_raw.isna().sum().values,
    "dtype": beach_raw.dtypes.astype(str).values
})

display(column_check)

Loaded file: sd_beach_advisories_clean_2023_latest.csv
Rows: 1166
Columns: 30


,start_date,end_date,duration_days,post_emergency,analysis_period,study_group,impact_level,advisory_type,advisory_cause,source,...,Station_LowerLon,beach_lat,beach_lon,Beach_LowerLat,Beach_LowerLon,enterococcus_flag,fecal_coliform_flag,total_coliform_flag,ecoli_flag,Ratio
0,2023-01-01,2023-01-26,25.0,0,Pre-emergency,Comparison Beaches,1,Posting,Bacterial Standards Violation,Unknown,...,-117.262800,32.807900,-117.266200,32.800800,-117.259400,1.0,0.0,0.0,0.0,0.0
1,2023-01-01,2023-07-10,190.0,0,Pre-emergency,South Bay / Tijuana River Impact,2,Closure,Other - Tijuana River Associated,Sewage/Grease,...,-117.133205,32.587527,-117.132598,32.566219,-117.133006,1.0,0.0,0.0,0.0,0.0
2,2023-01-01,2023-02-20,50.0,0,Pre-emergency,South Bay / Tijuana River Impact,2,Closure,Other - Tijuana River Associated,Sewage/Grease,...,-117.144500,32.636836,-117.144303,32.608936,-117.134783,1.0,0.0,0.0,0.0,0.0
3,2023-01-01,2023-01-09,8.0,0,Pre-emergency,Other San Diego Beaches,1,Rain,Rain,Rain,...,-117.124000,33.386900,-117.596000,32.534400,-117.124000,0.0,0.0,0.0,0.0,0.0
4,2023-01-01,2023-12-31,364.0,0,Pre-emergency,South Bay / Tijuana River Impact,2,Closure,Other - Tijuana River Associated,Sewage/Grease,...,-117.124300,32.552820,-117.127708,32.534367,-117.124197,1.0,0.0,0.0,0.0,0.0


,column,non_missing,missing,dtype
0,start_date,1166,0,object
1,end_date,1152,14,object
2,duration_days,1152,14,float64
3,post_emergency,1166,0,int64
4,analysis_period,1166,0,object
5,study_group,1166,0,object
6,impact_level,1166,0,int64
7,advisory_type,1166,0,object
8,advisory_cause,1142,24,object
9,source,1141,25,object


In [ ]:
missing_end = beach_raw[beach_raw["end_date"].isna()].copy()

print("Rows with missing end_date:", missing_end.shape[0])

display(
    missing_end[
        [
            "start_date",
            "end_date",
            "duration_days",
            "analysis_period",
            "study_group",
            "advisory_type",
            "advisory_cause",
            "source",
            "beach_name",
            "station_id",
            "station_name"
        ]
    ].sort_values(["start_date", "beach_name"])
)

print("\nMissing end_date by advisory_type:")
display(missing_end["advisory_type"].value_counts(dropna=False))

print("\nMissing end_date by study_group:")
display(missing_end["study_group"].value_counts(dropna=False))

print("\nMissing end_date by advisory_cause:")
display(missing_end["advisory_cause"].value_counts(dropna=False))


Rows with missing end_date: 14


,start_date,end_date,duration_days,analysis_period,study_group,advisory_type,advisory_cause,source,beach_name,station_id,station_name
28,2023-02-17,NaN,NaN,Pre-emergency,Comparison Beaches,Posting,Bacterial Standards Violation,.,"Mission Bay, Campland On The Bay",496,MB-080
1146,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,Border Field State Park,455,IB-010
1137,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,Coronado City beaches,325,EH-050
1139,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,Coronado City beaches,1038,IB-079
1138,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,Coronado north beach,328,EH-060
1141,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,"Imperial Beach municipal beach, other",319,EH-010
1144,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,"Imperial Beach municipal beach, other",459,IB-050
1145,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,Imperial Beach pier area,322,EH-030
1140,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,Silver Strand State Beach,461,IB-070
1142,2026-01-01,NaN,NaN,Recent follow-up,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Sewage/Grease,north Imperial Beach,460,IB-060



Missing end_date by advisory_type:


,count
advisory_type,
Closure,9
Posting,5



Missing end_date by study_group:


,count
study_group,
South Bay / Tijuana River Impact,9
Comparison Beaches,4
Other San Diego Beaches,1



Missing end_date by advisory_cause:


,count
advisory_cause,
Other - Tijuana River Associated,9
Bacterial Standards Violation,5


In [ ]:
beach_events = beach_raw.copy()

# Parse date columns
beach_events["start_date"] = pd.to_datetime(beach_events["start_date"], errors="coerce")
beach_events["end_date"] = pd.to_datetime(beach_events["end_date"], errors="coerce")

# Use rainfall project coverage as the project end date
PROJECT_START = pd.Timestamp("2023-01-01")
PROJECT_END = pd.Timestamp("2026-03-04")

# Flag missing end dates before filling
beach_events["end_date_was_missing"] = beach_events["end_date"].isna()

# Rule 1:
# Older missing end dates are treated as one-day records so they do not overcount activity.
older_missing = (
    beach_events["end_date"].isna()
    & (beach_events["start_date"] < pd.Timestamp("2026-01-01"))
)

beach_events.loc[older_missing, "end_date"] = beach_events.loc[older_missing, "start_date"]

# Rule 2:
# Recent 2026 missing end dates are treated as active through the project end date.
recent_missing = (
    beach_events["end_date"].isna()
    & (beach_events["start_date"] >= pd.Timestamp("2026-01-01"))
)

beach_events.loc[recent_missing, "end_date"] = PROJECT_END

# Recalculate duration after filling end dates
beach_events["duration_days_clean"] = (
    beach_events["end_date"] - beach_events["start_date"]
).dt.days

# Keep only valid records inside the project period
beach_events = beach_events[
    (beach_events["start_date"].notna())
    & (beach_events["end_date"].notna())
    & (beach_events["end_date"] >= PROJECT_START)
    & (beach_events["start_date"] <= PROJECT_END)
].copy()

# Clip date ranges to the project period
beach_events["start_date_clipped"] = beach_events["start_date"].clip(lower=PROJECT_START, upper=PROJECT_END)
beach_events["end_date_clipped"] = beach_events["end_date"].clip(lower=PROJECT_START, upper=PROJECT_END)

print("Rows after date cleaning:", beach_events.shape[0])
print("Project start:", PROJECT_START.date())
print("Project end:", PROJECT_END.date())
print("Records with originally missing end_date:", beach_events["end_date_was_missing"].sum())

display(
    beach_events.loc[
        beach_events["end_date_was_missing"],
        [
            "start_date",
            "end_date",
            "start_date_clipped",
            "end_date_clipped",
            "duration_days",
            "duration_days_clean",
            "study_group",
            "advisory_type",
            "advisory_cause",
            "beach_name",
            "station_id",
            "station_name"
        ]
    ].sort_values(["start_date", "beach_name"])
)

Rows after date cleaning: 1166
Project start: 2023-01-01
Project end: 2026-03-04
Records with originally missing end_date: 14


,start_date,end_date,start_date_clipped,end_date_clipped,duration_days,duration_days_clean,study_group,advisory_type,advisory_cause,beach_name,station_id,station_name
28,2023-02-17,2023-02-17,2023-02-17,2023-02-17,NaN,0,Comparison Beaches,Posting,Bacterial Standards Violation,"Mission Bay, Campland On The Bay",496,MB-080
1146,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Border Field State Park,455,IB-010
1137,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Coronado City beaches,325,EH-050
1139,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Coronado City beaches,1038,IB-079
1138,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Coronado north beach,328,EH-060
1141,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,"Imperial Beach municipal beach, other",319,EH-010
1144,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,"Imperial Beach municipal beach, other",459,IB-050
1145,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Imperial Beach pier area,322,EH-030
1140,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,Silver Strand State Beach,461,IB-070
1142,2026-01-01,2026-03-04,2026-01-01,2026-03-04,NaN,62,South Bay / Tijuana River Impact,Closure,Other - Tijuana River Associated,north Imperial Beach,460,IB-060


In [ ]:
bacteria_cols = [
    "enterococcus_flag",
    "fecal_coliform_flag",
    "total_coliform_flag",
    "ecoli_flag"
]

# Create a bacteria-related event flag.
# This uses both advisory cause and bacteria indicator columns.
beach_events["bacteria_related_event"] = (
    beach_events["advisory_cause"].astype(str).str.contains(
        "bacterial|bacteria|standards violation", case=False, na=False
    )
    | (beach_events[bacteria_cols].fillna(0).sum(axis=1) > 0)
)

# Create a Tijuana/sewage-related event flag for comparison.
beach_events["tijuana_sewage_related_event"] = (
    beach_events["advisory_cause"].astype(str).str.contains(
        "tijuana|river", case=False, na=False
    )
    | beach_events["source"].astype(str).str.contains(
        "sewage|grease", case=False, na=False
    )
)

print("Bacteria-related event records:", int(beach_events["bacteria_related_event"].sum()))
print("Tijuana/sewage-related event records:", int(beach_events["tijuana_sewage_related_event"].sum()))

print("\nBacteria flag totals:")
display(
    beach_events[bacteria_cols]
    .fillna(0)
    .sum()
    .astype(int)
    .to_frame(name="event_count")
)

print("\nBacteria-related records by advisory type:")
display(
    pd.crosstab(
        beach_events["advisory_type"],
        beach_events["bacteria_related_event"],
        margins=True
    )
)

print("\nBacteria-related records by study group:")
display(
    pd.crosstab(
        beach_events["study_group"],
        beach_events["bacteria_related_event"],
        margins=True
    )
)

print("\nAdvisory cause summary:")
display(
    beach_events["advisory_cause"]
    .value_counts(dropna=False)
    .to_frame(name="event_count")
)

Bacteria-related event records: 1028
Tijuana/sewage-related event records: 243

Bacteria flag totals:


,event_count
enterococcus_flag,980
fecal_coliform_flag,14
total_coliform_flag,13
ecoli_flag,2



Bacteria-related records by advisory type:


bacteria_related_event,False,True,All
advisory_type,,,
Closure,100,157,257
Posting,4,871,875
Rain,34,0,34
All,138,1028,1166



Bacteria-related records by study group:


bacteria_related_event,False,True,All
study_group,,,
Comparison Beaches,18,291,309
Other San Diego Beaches,56,292,348
South Bay / Tijuana River Impact,64,445,509
All,138,1028,1166



Advisory cause summary:


,event_count
advisory_cause,
Bacterial Standards Violation,876
Other - Tijuana River Associated,205
Sewage Spill,42
NaN,24
Rain,10
Blockage,2
Other - Dredging,2
Pump Station Failure,2
Other,1


In [ ]:
#getting ahead to when we need to link all our dataset
# need to build summary ready columns that can be expanded into daily
# beach activity records


beach_events = beach_events.copy()

# Clean text columns
beach_events["advisory_type_clean"] = beach_events["advisory_type"].astype(str).str.strip()
beach_events["advisory_cause_clean"] = beach_events["advisory_cause"].astype(str).str.strip()
beach_events["source_clean"] = beach_events["source"].astype(str).str.strip()
beach_events["beach_name_clean"] = beach_events["beach_name"].astype(str).str.strip()
beach_events["station_name_clean"] = beach_events["station_name"].astype(str).str.strip()

# Bacteria indicator columns
bacteria_cols = [
    "enterococcus_flag",
    "fecal_coliform_flag",
    "total_coliform_flag",
    "ecoli_flag"
]

# Make sure bacteria flags are numeric and clean
for col in bacteria_cols:
    beach_events[col] = pd.to_numeric(beach_events[col], errors="coerce").fillna(0).astype(int)

# Advisory type flags
beach_events["closure_active_day"] = (
    beach_events["advisory_type_clean"].str.lower().eq("closure")
).astype(int)

beach_events["posting_active_day"] = (
    beach_events["advisory_type_clean"].str.lower().eq("posting")
).astype(int)

beach_events["rain_active_day"] = (
    beach_events["advisory_type_clean"].str.lower().eq("rain")
    | beach_events["advisory_cause_clean"].str.contains("rain", case=False, na=False)
    | beach_events["source_clean"].str.contains("rain", case=False, na=False)
).astype(int)

# Any active advisory/closure/posting/rain record
beach_events["active_advisory_day"] = 1

# Bacteria-related event:
# Uses either the bacterial standards cause or one of the bacteria flags.
beach_events["bacteria_related_event"] = (
    beach_events["advisory_cause_clean"].str.contains(
        "bacterial standards violation|bacteria|bacterial",
        case=False,
        na=False
    )
    | (beach_events[bacteria_cols].sum(axis=1) > 0)
).astype(int)

# Tijuana River / sewage-related event:
# Includes Tijuana River, sewage, grease, spills, blockages, pump station failures, and line breaks.
sewage_pattern = "tijuana|sewage|grease|spill|blockage|pump station|line break|creek release"

beach_events["tijuana_sewage_related_event"] = (
    beach_events["advisory_cause_clean"].str.contains(sewage_pattern, case=False, na=False)
    | beach_events["source_clean"].str.contains(sewage_pattern, case=False, na=False)
).astype(int)

# Cause group for clean tables and paper interpretation.
def classify_cause_group(row):
    if row["tijuana_sewage_related_event"] == 1:
        return "Tijuana River / sewage associated"
    elif row["bacteria_related_event"] == 1:
        return "Bacteria-related"
    elif row["rain_active_day"] == 1:
        return "Rain advisory"
    else:
        return "Other / unknown"

beach_events["cause_group"] = beach_events.apply(classify_cause_group, axis=1)

# Severity score:
# Closure = 2, Posting = 1, Rain/Other = 0
# This is descriptive, not a measured health-risk score.
beach_events["severity_score"] = np.select(
    [
        beach_events["closure_active_day"] == 1,
        beach_events["posting_active_day"] == 1
    ],
    [
        2,
        1
    ],
    default=0
).astype(int)

print("Event flag summary:")
display(
    beach_events[
        [
            "active_advisory_day",
            "closure_active_day",
            "posting_active_day",
            "rain_active_day",
            "bacteria_related_event",
            "tijuana_sewage_related_event",
            "severity_score"
        ]
    ].sum().to_frame("event_count")
)

print("\nCause group counts:")
display(beach_events["cause_group"].value_counts(dropna=False).to_frame("event_count"))

print("\nCause group by advisory type:")
display(
    pd.crosstab(
        beach_events["cause_group"],
        beach_events["advisory_type_clean"],
        margins=True
    )
)

print("\nCause group by study group:")
display(
    pd.crosstab(
        beach_events["study_group"],
        beach_events["cause_group"],
        margins=True
    )
)


Event flag summary:


,event_count
active_advisory_day,1166
closure_active_day,257
posting_active_day,875
rain_active_day,43
bacteria_related_event,1028
tijuana_sewage_related_event,254
severity_score,1389



Cause group counts:


,event_count
cause_group,
Bacteria-related,876
Tijuana River / sewage associated,254
Rain advisory,34
Other / unknown,2



Cause group by advisory type:


advisory_type_clean,Closure,Posting,Rain,All
cause_group,,,,
Bacteria-related,7,869,0,876
Other / unknown,1,1,0,2
Rain advisory,0,0,34,34
Tijuana River / sewage associated,249,5,0,254
All,257,875,34,1166



Cause group by study group:


cause_group,Bacteria-related,Other / unknown,Rain advisory,Tijuana River / sewage associated,All
study_group,,,,,
Comparison Beaches,289,0,1,19,309
Other San Diego Beaches,290,2,33,23,348
South Bay / Tijuana River Impact,297,0,0,212,509
All,876,2,34,254,1166


Before expanding to daily rows. The only thing I want to inspect is why rain_active_day = 43 but cause_group = Rain advisory is only 34. That may be okay because some records have rain in source or cause, but were grouped into another higher-priority cause group. We just need to understand it.

In [ ]:
# session ended on colab so i had to reconnect everything and still make
# the bacteria rain/sewage flags, and adds severity score


from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np

DATA_RAW = Path("/content/drive/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"
FIGURES = DATA_RAW / "figures"
TABLES = DATA_RAW / "tables"

for folder in [DATA_PROCESSED, FIGURES, TABLES]:
    folder.mkdir(parents=True, exist_ok=True)

# Load cleaned event-level beach advisory file
beach_path = DATA_RAW / "sd_beach_advisories_clean_2023_latest.csv"

if not beach_path.exists():
    raise FileNotFoundError(f"Could not find: {beach_path}")

beach_events = pd.read_csv(beach_path, low_memory=False)

# Parse dates
beach_events["start_date"] = pd.to_datetime(beach_events["start_date"], errors="coerce")
beach_events["end_date"] = pd.to_datetime(beach_events["end_date"], errors="coerce")

# Project period based on completed rainfall file
PROJECT_START = pd.Timestamp("2023-01-01")
PROJECT_END = pd.Timestamp("2026-03-04")

# Flag missing end dates before filling
beach_events["end_date_was_missing"] = beach_events["end_date"].isna()

# Older missing end dates: conservative one-day event
older_missing = (
    beach_events["end_date"].isna()
    & (beach_events["start_date"] < pd.Timestamp("2026-01-01"))
)

beach_events.loc[older_missing, "end_date"] = beach_events.loc[older_missing, "start_date"]

# Recent 2026 missing end dates: active through project end
recent_missing = (
    beach_events["end_date"].isna()
    & (beach_events["start_date"] >= pd.Timestamp("2026-01-01"))
)

beach_events.loc[recent_missing, "end_date"] = PROJECT_END

# Recalculate duration
beach_events["duration_days_clean"] = (
    beach_events["end_date"] - beach_events["start_date"]
).dt.days

# Keep valid records inside project period
beach_events = beach_events[
    (beach_events["start_date"].notna())
    & (beach_events["end_date"].notna())
    & (beach_events["end_date"] >= PROJECT_START)
    & (beach_events["start_date"] <= PROJECT_END)
].copy()

# Clip to project period
beach_events["start_date_clipped"] = beach_events["start_date"].clip(lower=PROJECT_START, upper=PROJECT_END)
beach_events["end_date_clipped"] = beach_events["end_date"].clip(lower=PROJECT_START, upper=PROJECT_END)

# Clean text fields
beach_events["advisory_type_clean"] = beach_events["advisory_type"].astype(str).str.strip()
beach_events["advisory_cause_clean"] = beach_events["advisory_cause"].fillna("Unknown").astype(str).str.strip()
beach_events["source_clean"] = beach_events["source"].fillna("Unknown").astype(str).str.strip()
beach_events["beach_name_clean"] = beach_events["beach_name"].astype(str).str.strip()
beach_events["station_name_clean"] = beach_events["station_name"].astype(str).str.strip()

# Bacteria columns
bacteria_cols = [
    "enterococcus_flag",
    "fecal_coliform_flag",
    "total_coliform_flag",
    "ecoli_flag"
]

for col in bacteria_cols:
    beach_events[col] = pd.to_numeric(beach_events[col], errors="coerce").fillna(0).astype(int)

# Advisory type flags
beach_events["closure_active_day"] = (
    beach_events["advisory_type_clean"].str.lower().eq("closure")
).astype(int)

beach_events["posting_active_day"] = (
    beach_events["advisory_type_clean"].str.lower().eq("posting")
).astype(int)

# Exact rain flag only, avoids false match with "Stormdrain"
beach_events["rain_advisory_event"] = (
    beach_events["advisory_type_clean"].str.fullmatch("Rain", case=False, na=False)
    | beach_events["advisory_cause_clean"].str.fullmatch("Rain", case=False, na=False)
    | beach_events["source_clean"].str.fullmatch("Rain", case=False, na=False)
).astype(int)

# Every row represents an active advisory/posting/closure event before expansion
beach_events["active_advisory_day"] = 1

# Bacteria-related event
beach_events["bacteria_related_event"] = (
    beach_events["advisory_cause_clean"].str.contains(
        "bacterial standards violation|bacteria|bacterial",
        case=False,
        na=False
    )
    | (beach_events[bacteria_cols].sum(axis=1) > 0)
).astype(int)

# Tijuana River-specific flag
beach_events["tijuana_river_related_event"] = (
    beach_events["advisory_cause_clean"].str.contains("tijuana", case=False, na=False)
).astype(int)

# Other sewage infrastructure flag
sewage_infra_pattern = "sewage spill|sewer line|blockage|pump station|line break|creek release"

beach_events["sewage_infrastructure_event"] = (
    beach_events["advisory_cause_clean"].str.contains(sewage_infra_pattern, case=False, na=False)
    | beach_events["source_clean"].str.contains(sewage_infra_pattern, case=False, na=False)
).astype(int)

# Broad sewage-related flag
beach_events["sewage_related_event"] = (
    (beach_events["tijuana_river_related_event"] == 1)
    | (beach_events["sewage_infrastructure_event"] == 1)
).astype(int)

# Mutually exclusive cause group
def classify_cause_group(row):
    if row["tijuana_river_related_event"] == 1:
        return "Tijuana River associated"
    elif row["sewage_infrastructure_event"] == 1:
        return "Other sewage infrastructure"
    elif row["bacteria_related_event"] == 1:
        return "Bacteria-related"
    elif row["rain_advisory_event"] == 1:
        return "Rain advisory"
    else:
        return "Other / unknown"

beach_events["cause_group"] = beach_events.apply(classify_cause_group, axis=1)

# Severity score: descriptive intensity only
beach_events["severity_score"] = np.select(
    [
        beach_events["closure_active_day"] == 1,
        beach_events["posting_active_day"] == 1
    ],
    [
        2,
        1
    ],
    default=0
).astype(int)

print("beach_events rebuilt successfully.")
print("Rows:", beach_events.shape[0])
print("Date range:", beach_events["start_date_clipped"].min().date(), "to", beach_events["end_date_clipped"].max().date())
print("Originally missing end_date records:", int(beach_events["end_date_was_missing"].sum()))

display(beach_events.head())

Mounted at /content/drive
beach_events rebuilt successfully.
Rows: 1166
Date range: 2023-01-01 to 2026-03-04
Originally missing end_date records: 14


,start_date,end_date,duration_days,post_emergency,analysis_period,study_group,impact_level,advisory_type,advisory_cause,source,...,closure_active_day,posting_active_day,rain_advisory_event,active_advisory_day,bacteria_related_event,tijuana_river_related_event,sewage_infrastructure_event,sewage_related_event,cause_group,severity_score
0,2023-01-01,2023-01-26,25.0,0,Pre-emergency,Comparison Beaches,1,Posting,Bacterial Standards Violation,Unknown,...,0,1,0,1,1,0,0,0,Bacteria-related,1
1,2023-01-01,2023-07-10,190.0,0,Pre-emergency,South Bay / Tijuana River Impact,2,Closure,Other - Tijuana River Associated,Sewage/Grease,...,1,0,0,1,1,1,0,1,Tijuana River associated,2
2,2023-01-01,2023-02-20,50.0,0,Pre-emergency,South Bay / Tijuana River Impact,2,Closure,Other - Tijuana River Associated,Sewage/Grease,...,1,0,0,1,1,1,0,1,Tijuana River associated,2
3,2023-01-01,2023-01-09,8.0,0,Pre-emergency,Other San Diego Beaches,1,Rain,Rain,Rain,...,0,0,1,1,0,0,0,0,Rain advisory,0
4,2023-01-01,2023-12-31,364.0,0,Pre-emergency,South Bay / Tijuana River Impact,2,Closure,Other - Tijuana River Associated,Sewage/Grease,...,1,0,0,1,1,1,0,1,Tijuana River associated,2


In [ ]:
# still feel that it would look better if i summarize flag/groups and study groups patterns

print("Improved event flag summary:")
display(
    beach_events[
        [
            "active_advisory_day",
            "closure_active_day",
            "posting_active_day",
            "rain_advisory_event",
            "bacteria_related_event",
            "tijuana_river_related_event",
            "sewage_infrastructure_event",
            "sewage_related_event",
            "severity_score"
        ]
    ].sum().to_frame("event_count")
)

print("\nImproved cause group counts:")
display(beach_events["cause_group"].value_counts(dropna=False).to_frame("event_count"))

Improved event flag summary:


,event_count
active_advisory_day,1166
closure_active_day,257
posting_active_day,875
rain_advisory_event,34
bacteria_related_event,1028
tijuana_river_related_event,205
sewage_infrastructure_event,48
sewage_related_event,253
severity_score,1389



Improved cause group counts:


,event_count
cause_group,
Bacteria-related,877
Tijuana River associated,205
Other sewage infrastructure,48
Rain advisory,34
Other / unknown,2


rain_advisory_event is now 34, so the false “Stormdrain = rain” issue is fixed.
tijuana_river_related_event is 205, which keeps Tijuana River records separate.
sewage_infrastructure_event is 48, which captures other sewage/sewer issues outside the Tijuana River category.
sewage_related_event is 253, which combines Tijuana River + other sewage-related records.
cause_group adds up correctly to 1,166 total records.

One thing to understand: bacteria_related_event = 1028, but cause_group = Bacteria-related is only 877 because some records are both bacteria-related and sewage-related. For the mutually exclusive cause_group, we are prioritizing Tijuana/sewage first. That is the right choice for this project because the main project focus is sewage-related exposur

In [ ]:
# now its taking shape were we expand each advisory/closure event across its active date range
# and keeps clean flags for daily analysis.


beach_events = beach_events.reset_index(drop=True).copy()
beach_events["beach_event_id"] = beach_events.index + 1

# Columns to keep in the long daily table
keep_cols = [
    "beach_event_id",
    "start_date",
    "end_date",
    "start_date_clipped",
    "end_date_clipped",
    "duration_days_clean",
    "analysis_period",
    "post_emergency",
    "study_group",
    "impact_level",
    "advisory_type_clean",
    "advisory_cause_clean",
    "source_clean",
    "cause_group",
    "beach_name_clean",
    "station_id",
    "station_name_clean",
    "station_lat",
    "station_lon",
    "beach_lat",
    "beach_lon",
    "active_advisory_day",
    "closure_active_day",
    "posting_active_day",
    "rain_advisory_event",
    "bacteria_related_event",
    "tijuana_river_related_event",
    "sewage_infrastructure_event",
    "sewage_related_event",
    "severity_score",
    "enterococcus_flag",
    "fecal_coliform_flag",
    "total_coliform_flag",
    "ecoli_flag",
    "end_date_was_missing"
]

# Keep only columns that exist, just in case the file changes later
keep_cols = [col for col in keep_cols if col in beach_events.columns]

# Expand each event into one row per active date
beach_daily_long = beach_events[keep_cols].copy()

beach_daily_long["date"] = beach_daily_long.apply(
    lambda row: pd.date_range(row["start_date_clipped"], row["end_date_clipped"], freq="D"),
    axis=1
)

beach_daily_long = beach_daily_long.explode("date").reset_index(drop=True)

# Put date first for readability
date_col = beach_daily_long.pop("date")
beach_daily_long.insert(0, "date", date_col)

print("Original event records:", beach_events.shape[0])
print("Expanded daily active records:", beach_daily_long.shape[0])
print("Date range:", beach_daily_long["date"].min().date(), "to", beach_daily_long["date"].max().date())
print("Unique beaches:", beach_daily_long["beach_name_clean"].nunique())
print("Unique stations:", beach_daily_long["station_id"].nunique())

display(beach_daily_long.head())


Original event records: 1166
Expanded daily active records: 12915
Date range: 2023-01-01 to 2026-03-04
Unique beaches: 48
Unique stations: 106


,date,beach_event_id,start_date,end_date,start_date_clipped,end_date_clipped,duration_days_clean,analysis_period,post_emergency,study_group,...,bacteria_related_event,tijuana_river_related_event,sewage_infrastructure_event,sewage_related_event,severity_score,enterococcus_flag,fecal_coliform_flag,total_coliform_flag,ecoli_flag,end_date_was_missing
0,2023-01-01,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False
1,2023-01-02,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False
2,2023-01-03,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False
3,2023-01-04,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False
4,2023-01-05,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False


In [ ]:
beach_events["expected_active_days"] = (
    beach_events["end_date_clipped"] - beach_events["start_date_clipped"]
).dt.days + 1

expected_total_rows = beach_events["expected_active_days"].sum()
actual_total_rows = beach_daily_long.shape[0]

print("Expected expanded rows:", expected_total_rows)
print("Actual expanded rows:", actual_total_rows)
print("Expansion matches expected:", expected_total_rows == actual_total_rows)

# Check each event expanded to the expected number of daily rows
event_expansion_check = (
    beach_daily_long
    .groupby("beach_event_id")
    .agg(actual_active_days=("date", "nunique"))
    .reset_index()
    .merge(
        beach_events[["beach_event_id", "expected_active_days", "beach_name_clean", "advisory_type_clean", "cause_group"]],
        on="beach_event_id",
        how="left"
    )
)

event_expansion_check["day_difference"] = (
    event_expansion_check["actual_active_days"] - event_expansion_check["expected_active_days"]
)

print("\nEvent-level expansion problems:")
display(
    event_expansion_check[event_expansion_check["day_difference"] != 0]
)

print("\nDaily long table date coverage:")
display(
    pd.DataFrame({
        "metric": [
            "minimum_date",
            "maximum_date",
            "total_active_daily_records",
            "unique_dates",
            "unique_beaches",
            "unique_stations"
        ],
        "value": [
            beach_daily_long["date"].min(),
            beach_daily_long["date"].max(),
            beach_daily_long.shape[0],
            beach_daily_long["date"].nunique(),
            beach_daily_long["beach_name_clean"].nunique(),
            beach_daily_long["station_id"].nunique()
        ]
    })
)

print("\nDaily long activity by cause group:")
display(
    beach_daily_long["cause_group"]
    .value_counts()
    .to_frame("active_record_days")
)

print("\nDaily long activity by study group:")
display(
    beach_daily_long["study_group"]
    .value_counts()
    .to_frame("active_record_days")
)

Expected expanded rows: 12915
Actual expanded rows: 12915
Expansion matches expected: True

Event-level expansion problems:


,beach_event_id,actual_active_days,expected_active_days,beach_name_clean,advisory_type_clean,cause_group,day_difference



Daily long table date coverage:


,metric,value
0,minimum_date,2023-01-01 00:00:00
1,maximum_date,2026-03-04 00:00:00
2,total_active_daily_records,12915
3,unique_dates,1159
4,unique_beaches,48
5,unique_stations,106



Daily long activity by cause group:


,active_record_days
cause_group,
Bacteria-related,6678
Tijuana River associated,5768
Other sewage infrastructure,266
Rain advisory,199
Other / unknown,4



Daily long activity by study group:


,active_record_days
study_group,
South Bay / Tijuana River Impact,7168
Other San Diego Beaches,2980
Comparison Beaches,2767


Final Notes for this coding session:

After expanding advisory events into active daily records, South Bay / Tijuana River Impact beaches had the highest number of active record-days: 7,168 compared with 2,980 for Other San Diego Beaches and 2,767 for Comparison Beaches. Tijuana River associated records also made up 5,768 active record-days, showing that long-duration closures are a major part of the beach advisory pattern.

Important wording: use active record-days, not “number of closures,” because long events create multiple daily records.

In [ ]:
# creating a notebook session on my drive to make sure everything is organize and
# i have proof of all my work

NOTEBOOKS = DATA_RAW / "notebooks"
NOTEBOOKS.mkdir(parents=True, exist_ok=True)

print("Notebook folder:", NOTEBOOKS)

Notebook folder: /content/drive/MyDrive/STAT596/Project596_datafiles/notebooks


In [ ]:

long_output_path = DATA_PROCESSED / "daily_beach_activity_long.csv"

beach_daily_long.to_csv(long_output_path, index=False)

print("Saved long daily beach activity file:")
print(long_output_path)
print("Rows:", beach_daily_long.shape[0])
print("Columns:", beach_daily_long.shape[1])

Saved long daily beach activity file:
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/daily_beach_activity_long.csv
Rows: 12915
Columns: 36


In [ ]:
# creating daily beach closure csv file


from pathlib import Path
import pandas as pd
import numpy as np

DATA_RAW = Path("/content/drive/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"

long_path = DATA_PROCESSED / "daily_beach_activity_long.csv"
daily_output_path = DATA_PROCESSED / "daily_beach_closure_activity.csv"

# Load saved long-format beach activity file
beach_daily_long = pd.read_csv(long_path)
beach_daily_long["date"] = pd.to_datetime(beach_daily_long["date"])
beach_daily_long["start_date_clipped"] = pd.to_datetime(beach_daily_long["start_date_clipped"])

# Make sure flag columns are numeric
flag_cols = [
    "active_advisory_day",
    "closure_active_day",
    "posting_active_day",
    "rain_advisory_event",
    "bacteria_related_event",
    "tijuana_river_related_event",
    "sewage_infrastructure_event",
    "sewage_related_event",
    "severity_score",
    "enterococcus_flag",
    "fecal_coliform_flag",
    "total_coliform_flag",
    "ecoli_flag"
]

for col in flag_cols:
    beach_daily_long[col] = pd.to_numeric(
        beach_daily_long[col],
        errors="coerce"
    ).fillna(0).astype(int)

# Complete project date range
PROJECT_START = beach_daily_long["date"].min()
PROJECT_END = beach_daily_long["date"].max()

all_dates = pd.DataFrame({
    "date": pd.date_range(PROJECT_START, PROJECT_END, freq="D")
})

# Main daily summary: one row per date
daily_beach = (
    beach_daily_long
    .groupby("date")
    .agg(
        active_advisory_records=("active_advisory_day", "sum"),
        active_closure_records=("closure_active_day", "sum"),
        active_posting_records=("posting_active_day", "sum"),
        active_rain_advisory_records=("rain_advisory_event", "sum"),
        active_bacteria_related_records=("bacteria_related_event", "sum"),
        active_tijuana_river_related_records=("tijuana_river_related_event", "sum"),
        active_sewage_infrastructure_records=("sewage_infrastructure_event", "sum"),
        active_sewage_related_records=("sewage_related_event", "sum"),
        active_enterococcus_records=("enterococcus_flag", "sum"),
        active_fecal_coliform_records=("fecal_coliform_flag", "sum"),
        active_total_coliform_records=("total_coliform_flag", "sum"),
        active_ecoli_records=("ecoli_flag", "sum"),
        daily_severity_score=("severity_score", "sum"),
        max_severity_score=("severity_score", "max"),
        unique_beaches_impacted=("beach_name_clean", "nunique"),
        unique_stations_impacted=("station_id", "nunique")
    )
    .reset_index()
)

# Create event-level table from the long file so new records are not overcounted
event_unique = (
    beach_daily_long
    .drop_duplicates("beach_event_id")
    .copy()
)

daily_new = (
    event_unique
    .groupby("start_date_clipped")
    .agg(
        new_advisory_records=("beach_event_id", "nunique"),
        new_closure_records=("closure_active_day", "sum"),
        new_posting_records=("posting_active_day", "sum"),
        new_rain_advisory_records=("rain_advisory_event", "sum"),
        new_bacteria_related_records=("bacteria_related_event", "sum"),
        new_tijuana_river_related_records=("tijuana_river_related_event", "sum"),
        new_sewage_related_records=("sewage_related_event", "sum")
    )
    .reset_index()
    .rename(columns={"start_date_clipped": "date"})
)

# Study group daily summaries
def clean_group_name(text):
    text = str(text).lower()
    text = text.replace("/", " ")
    text = text.replace("-", " ")
    text = "_".join(text.split())
    return text

study_group_daily = (
    beach_daily_long
    .groupby(["date", "study_group"])
    .agg(
        active_records=("active_advisory_day", "sum"),
        closure_records=("closure_active_day", "sum"),
        severity_score=("severity_score", "sum"),
        unique_beaches=("beach_name_clean", "nunique")
    )
    .reset_index()
)

study_group_daily["study_group_clean"] = study_group_daily["study_group"].apply(clean_group_name)

active_wide = (
    study_group_daily
    .pivot(index="date", columns="study_group_clean", values="active_records")
    .add_prefix("active_records_")
    .reset_index()
)

closure_wide = (
    study_group_daily
    .pivot(index="date", columns="study_group_clean", values="closure_records")
    .add_prefix("closure_records_")
    .reset_index()
)

severity_wide = (
    study_group_daily
    .pivot(index="date", columns="study_group_clean", values="severity_score")
    .add_prefix("severity_score_")
    .reset_index()
)

beaches_wide = (
    study_group_daily
    .pivot(index="date", columns="study_group_clean", values="unique_beaches")
    .add_prefix("unique_beaches_")
    .reset_index()
)

# Merge all daily summaries
daily_beach = (
    all_dates
    .merge(daily_beach, on="date", how="left")
    .merge(daily_new, on="date", how="left")
    .merge(active_wide, on="date", how="left")
    .merge(closure_wide, on="date", how="left")
    .merge(severity_wide, on="date", how="left")
    .merge(beaches_wide, on="date", how="left")
)

# Fill missing counts with zero
count_cols = [col for col in daily_beach.columns if col != "date"]
daily_beach[count_cols] = daily_beach[count_cols].fillna(0).astype(int)

# Binary daily indicators
daily_beach["any_active_advisory"] = daily_beach["active_advisory_records"] > 0
daily_beach["any_active_closure"] = daily_beach["active_closure_records"] > 0
daily_beach["any_bacteria_related_activity"] = daily_beach["active_bacteria_related_records"] > 0
daily_beach["any_tijuana_river_related_activity"] = daily_beach["active_tijuana_river_related_records"] > 0
daily_beach["any_sewage_related_activity"] = daily_beach["active_sewage_related_records"] > 0

# Save final daily beach table
daily_beach.to_csv(daily_output_path, index=False)

print("Saved daily beach closure/activity file:")
print(daily_output_path)
print("Rows:", daily_beach.shape[0])
print("Columns:", daily_beach.shape[1])
print("Date range:", daily_beach["date"].min().date(), "to", daily_beach["date"].max().date())

display(daily_beach.head())

Saved daily beach closure/activity file:
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/daily_beach_closure_activity.csv
Rows: 1159
Columns: 41
Date range: 2023-01-01 to 2026-03-04


,date,active_advisory_records,active_closure_records,active_posting_records,active_rain_advisory_records,active_bacteria_related_records,active_tijuana_river_related_records,active_sewage_infrastructure_records,active_sewage_related_records,active_enterococcus_records,...,severity_score_other_san_diego_beaches,severity_score_south_bay_tijuana_river_impact,unique_beaches_comparison_beaches,unique_beaches_other_san_diego_beaches,unique_beaches_south_bay_tijuana_river_impact,any_active_advisory,any_active_closure,any_bacteria_related_activity,any_tijuana_river_related_activity,any_sewage_related_activity
0,2023-01-01,6,3,2,1,5,3,0,3,5,...,0,7,1,1,4,True,True,True,True,True
1,2023-01-02,7,4,2,1,6,4,0,4,6,...,0,9,1,1,4,True,True,True,True,True
2,2023-01-03,6,4,1,1,5,4,0,4,5,...,0,8,1,1,4,True,True,True,True,True
3,2023-01-04,6,4,1,1,5,4,0,4,5,...,0,8,1,1,4,True,True,True,True,True
4,2023-01-05,6,4,1,1,5,4,0,4,5,...,0,8,1,1,4,True,True,True,True,True


In [ ]:
# im double checking both my files were can because i didnt see them on my drive


files_to_check = {
    "daily_beach_activity_long": DATA_PROCESSED / "daily_beach_activity_long.csv",
    "daily_beach_closure_activity": DATA_PROCESSED / "daily_beach_closure_activity.csv"
}

for name, path in files_to_check.items():
    print("=" * 70)
    print(name)
    print("Path:", path)
    print("File exists:", path.exists())

    if path.exists():
        df_check = pd.read_csv(path)

        print("Rows:", df_check.shape[0])
        print("Columns:", df_check.shape[1])

        if "date" in df_check.columns:
            df_check["date"] = pd.to_datetime(df_check["date"], errors="coerce")
            print("Date range:", df_check["date"].min().date(), "to", df_check["date"].max().date())
            print("Unique dates:", df_check["date"].nunique())

        display(df_check.head())

daily_beach_activity_long
Path: /content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/daily_beach_activity_long.csv
File exists: True
Rows: 12915
Columns: 36
Date range: 2023-01-01 to 2026-03-04
Unique dates: 1159


,date,beach_event_id,start_date,end_date,start_date_clipped,end_date_clipped,duration_days_clean,analysis_period,post_emergency,study_group,...,bacteria_related_event,tijuana_river_related_event,sewage_infrastructure_event,sewage_related_event,severity_score,enterococcus_flag,fecal_coliform_flag,total_coliform_flag,ecoli_flag,end_date_was_missing
0,2023-01-01,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False
1,2023-01-02,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False
2,2023-01-03,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False
3,2023-01-04,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False
4,2023-01-05,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,1,0,0,0,1,1,0,0,0,False


daily_beach_closure_activity
Path: /content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/daily_beach_closure_activity.csv
File exists: True
Rows: 1159
Columns: 41
Date range: 2023-01-01 to 2026-03-04
Unique dates: 1159


,date,active_advisory_records,active_closure_records,active_posting_records,active_rain_advisory_records,active_bacteria_related_records,active_tijuana_river_related_records,active_sewage_infrastructure_records,active_sewage_related_records,active_enterococcus_records,...,severity_score_other_san_diego_beaches,severity_score_south_bay_tijuana_river_impact,unique_beaches_comparison_beaches,unique_beaches_other_san_diego_beaches,unique_beaches_south_bay_tijuana_river_impact,any_active_advisory,any_active_closure,any_bacteria_related_activity,any_tijuana_river_related_activity,any_sewage_related_activity
0,2023-01-01,6,3,2,1,5,3,0,3,5,...,0,7,1,1,4,True,True,True,True,True
1,2023-01-02,7,4,2,1,6,4,0,4,6,...,0,9,1,1,4,True,True,True,True,True
2,2023-01-03,6,4,1,1,5,4,0,4,5,...,0,8,1,1,4,True,True,True,True,True
3,2023-01-04,6,4,1,1,5,4,0,4,5,...,0,8,1,1,4,True,True,True,True,True
4,2023-01-05,6,4,1,1,5,4,0,4,5,...,0,8,1,1,4,True,True,True,True,True
